In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [18]:
df = pd.read_csv('coffee_data.csv')
df['agtron_whole'] = df['agtron'].str.extract(r'^(\d+)').astype(float)
print(f'Shape: {df.shape}')
print(f'Roast levels: {df["roast_level"].unique()}')
print('Missing values:')
print(df.isnull().sum())
df.head()

Shape: (8999, 8)
Roast levels: ['Light' 'Medium-Light' 'Medium' 'Medium-Dark' 'Dark' nan 'Very Dark']
Missing values:
name              0
score             7
roast_level     408
aroma           129
structure      5100
body             71
flavour          75
aftertaste      862
dtype: int64


,name,score,roast_level,aroma,structure,body,flavour,aftertaste
0,Papua New Guinea Chimbu,95.0,Light,9.0,9.0,9.0,9.0,9.0
1,Hong Kong Night,93.0,Medium-Light,9.0,NaN,8.0,9.0,8.0
2,Hong Kong Day,93.0,Medium-Light,8.0,NaN,9.0,9.0,8.0
3,Sunset Blend,92.0,Medium-Light,8.0,NaN,8.0,9.0,8.0
4,Ethiopia Sidama Yaye Natural — Special Project,96.0,Light,9.0,9.0,9.0,10.0,9.0


In [19]:
FEATURES = [
    'roast_level',
    'aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score', 'agtron_whole',
    'roaster', 'coffee_origin', 'roaster_location'
]
TARGET = 'name'

X = df[FEATURES].copy()
y = df[TARGET]

roast_order = [['Light', 'Medium-Light', 'Medium', 'Medium-Dark', 'Dark']]

preprocessor = ColumnTransformer(transformers=[
    ('roast', Pipeline([
        ('encode', OrdinalEncoder(categories=roast_order, handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scale', StandardScaler())
    ]), ['roast_level']),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), ['aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score', 'agtron_whole']),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), ['roaster', 'coffee_origin', 'roaster_location'])
])

print('Preprocessor ready.')

Preprocessor ready.


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

Train: 7199 samples | Test: 1800 samples


In [21]:
preprocessor.fit(X_train)
X_train_t = preprocessor.transform(X_train)
X_test_t  = preprocessor.transform(X_test)
y_train_arr = y_train.values
print('Data transformed.')

Data transformed.


In [22]:
def predict_cosine(X_query, X_ref, y_ref, top_n=5):
    """Return top_n coffee names ranked by mean cosine similarity across training examples."""
    sims = cosine_similarity(X_query, X_ref)[0]  # similarity to every training row

    # aggregate per coffee name using mean similarity
    # mean rather than sum so coffees with more training examples don't get an unfair boost
    name_sims = {}
    name_counts = {}
    for sim, name in zip(sims, y_ref):
        name_sims[name]   = name_sims.get(name, 0) + sim
        name_counts[name] = name_counts.get(name, 0) + 1

    mean_sims = {name: name_sims[name] / name_counts[name] for name in name_sims}
    ranked = sorted(mean_sims.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

print('predict_cosine() ready.')

predict_cosine() ready.


In [23]:
import math

# retrieve the full ranked list per query so MRR and NDCG can search beyond top-5
n_classes = len(np.unique(y_train_arr))

top1_count = top3_count = top5_count = top10_count = 0
mrr_total = ndcg_total = 0.0

for i in range(len(X_test_t)):
    query      = X_test_t[i:i+1]
    true_label = y_test.values[i]
    ranked     = predict_cosine(query, X_train_t, y_train_arr, top_n=n_classes)
    names      = [n for n, _ in ranked]

    if true_label == names[0]:    top1_count  += 1
    if true_label in names[:3]:   top3_count  += 1
    if true_label in names[:5]:   top5_count  += 1
    if true_label in names[:10]:  top10_count += 1

    if true_label in names:
        rank = names.index(true_label) + 1  # 1-indexed
        mrr_total  += 1.0 / rank
        if rank <= 10:
            ndcg_total += 1.0 / math.log2(rank + 1)  # IDCG=1 (ideal rank=1)

n = len(y_test)
print(f'Top-1  hit rate : {top1_count  / n:.4f}')
print(f'Top-3  hit rate : {top3_count  / n:.4f}')
print(f'Top-5  hit rate : {top5_count  / n:.4f}')
print(f'Top-10 hit rate : {top10_count / n:.4f}')
print(f'MRR             : {mrr_total   / n:.4f}')
print(f'NDCG@10         : {ndcg_total  / n:.4f}')

Top-1 accuracy: 0.001
Top-3 accuracy: 0.005
Top-5 accuracy: 0.006


In [24]:
def recommend_coffee(
        roast_level, aroma, structure, body, flavour, aftertaste,
        score=95, agtron_whole=None,
        roaster=None, coffee_origin=None, roaster_location=None,
        top_n=5):
    input_df = pd.DataFrame([{
        'roast_level':      roast_level,
        'aroma':            aroma,
        'structure':        structure,
        'body':             body,
        'flavour':          flavour,
        'aftertaste':       aftertaste,
        'score':            score,
        'agtron_whole':     agtron_whole,
        'roaster':          roaster,
        'coffee_origin':    coffee_origin,
        'roaster_location': roaster_location
    }])

    X_in   = preprocessor.transform(input_df)
    ranked = predict_cosine(X_in, X_train_t, y_train_arr, top_n=top_n)

    print(f'Attributes: roast={roast_level}, aroma={aroma}, structure={structure}, '
          f'body={body}, flavour={flavour}, aftertaste={aftertaste}, score={score}')
    print()
    print(f'Top {top_n} recommended coffees:')
    for rank, (name, sim) in enumerate(ranked, 1):
        print(f'  {rank}. {name}  (cosine similarity: {sim:.4f})')


# categorical params are optional — imputer fills most_frequent when None
recommend_coffee('Light', aroma=9, structure=9, body=9, flavour=10, aftertaste=9)

Attributes: roast=Light, aroma=9, structure=9, body=9, flavour=10, aftertaste=9, score=95

Top 5 recommended coffees:
  1. Panama Elida Natural Lot #13  (cosine similarity: 1.0000)
  2. Taiwan Natural Alishan Lalauya Zou Zhu Yuan Gesha  (cosine similarity: 0.9964)
  3. Wilton Benitez Colombia Yellow Bourbon  (cosine similarity: 0.9964)
  4. Ethiopia Ayla Bombe Washed  (cosine similarity: 0.9964)
  5. Panama Lerida Estate Ronella Lot 5 Washed Geisha  (cosine similarity: 0.9964)
